
###### 13_agent_tools

###### Purpose

Create reusable SQL and Vector Search tools and demonstrate rule-based tool routing based on the user’s question. The retrieved tool results can optionally be passed to an LLM to produce a grounded response.


###### Technologies Used

- Databricks

- Delta Lake

- Unity Catalog

- Databricks Vector Search

- Databricks Embedding Foundation Model (databricks-gte-large-en)

- LLM Model (databricks-meta-llama-3-1-8b-instruct)

- Python

- PySpark

- Databricks SDK

- Rule-Based Routing

###### Input

-  User question

-  Existing Vector Search endpoint and index

-  Customer-notes Delta table

-  Customer-note embeddings

-  Embedding and LLM endpoint configuration

######  Output

- Selected tool name

- Structured tool result

- Direct count response or LLM-generated grounded response

######  Architecture

```text

                         ┌─ SQL Count Tool ── Direct Answer
User Question → Router ──┤
                         └─ Vector Search Tool → Context → LLM → Answer

```


###### Section 0 : Installation

In [0]:
#Install Vector Search client
%pip install databricks-vectorsearch
dbutils.library.restartPython()

###### Section 1 :  Load Project Configuration

In [0]:
%run ./00_project_config

###### Section 2 : Import Libraries and Initialize Clients

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.serving import (
    ChatMessage,
    ChatMessageRole
)

w = WorkspaceClient()
vsc = VectorSearchClient(disable_notice=True)

###### Section 3 : Connect to Existing Vector Search Index

In [0]:
index = vsc.get_index(
    endpoint_name = VECTOR_SEARCH_ENDPOINT_NAME,
    index_name = VECTOR_INDEX_NAME
)

index_description = index.describe()

print("Connected to the existing Vector Search index.")
print(
    "Index state:",
    index_description
    .get("status", {})
    .get("detailed_state", "UNKNOWN")
)

###### Section 3 : Define Vector Search Tool

In [0]:
def search_customer_notes(
    question: str,
    num_results: int = 3
):
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    if num_results <= 0:
        raise ValueError(
            "num_results must be greater than zero."
        )

    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    if (
        not response.data
        or response.data[0].embedding is None
    ):
        raise ValueError(
            "The embedding model returned no embedding."
        )

    question_embedding = [
        float(value)
        for value in response.data[0].embedding
    ]

    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    rows = (
        results
        .get("result", {})
        .get("data_array", [])
    )

    if not rows:
        return {
            "tool": "search_customer_notes",
            "status": "no_results",
            "context": "",
            "rows": [],
            "result_count": 0
        }

    context_lines = [
        f"Customer {int(customer_id)}: {note}"
        for customer_id, note, score in rows
    ]

    return {
        "tool": "search_customer_notes",
        "status": "success",
        "context": "\n".join(context_lines),
        "rows": rows,
        "result_count": len(rows)
    }

###### Section 4 : Define SQL Analytics Tool

In [0]:
def count_customer_notes():
    count = spark.table(NOTES_TABLE).count()

    return {
        "tool": "count_customer_notes",
        "status": "success",
        "count": count
    }

###### Section 5 : Define Rule-Based Tool Router

In [0]:
def rule_based_agent(question: str):
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    question_lower = question.lower()

    count_phrases = [
        "how many",
        "number of",
        "total customer notes",
        "count customer notes"
    ]

    if any(
        phrase in question_lower
        for phrase in count_phrases
    ):
        return count_customer_notes()

    return search_customer_notes(question)

###### Section 6 : Unit-Test Individual Tools

In [0]:
count_test = count_customer_notes()

assert count_test["status"] == "success"
assert count_test["count"] > 0

print(count_test)

In [0]:
search_test = search_customer_notes(
    "Why are customers cancelling service?"
)

assert search_test["status"] == "success"
assert search_test["result_count"] > 0

print(search_test["context"])

###### Section 7 : Test Rule-Based Routing

In [0]:
count_route_test = rule_based_agent(
    "How many customer notes are there?"
)

assert count_route_test["tool"] == "count_customer_notes"

In [0]:
search_route_test = rule_based_agent(
    "Why are customers cancelling service?"
)

assert search_route_test["tool"] == "search_customer_notes"

###### Section 8 : Define LLM Response Helper

In [0]:
def generate_answer(prompt: str):
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=300,
        temperature=0.0
    )

    if (
        not response.choices
        or response.choices[0].message is None
    ):
        raise ValueError(
            "The LLM returned no response."
        )

    return response.choices[0].message.content

###### Section 9 : Generate Final Grounded Responses

In [0]:
def answer_with_tools(question: str):
    tool_result = rule_based_agent(question)

    if tool_result["tool"] == "count_customer_notes":
        return {
            "selected_tool": tool_result["tool"],
            "tool_result": tool_result,
            "answer": (
                f"There are {tool_result['count']} "
                "customer notes."
            )
        }

    if tool_result["status"] == "no_results":
        return {
            "selected_tool": tool_result["tool"],
            "tool_result": tool_result,
            "answer": (
                "I don't have enough information from "
                "the retrieved customer notes."
            )
        }

    context = tool_result["context"]

    prompt = f"""
You are a telecom customer-support assistant.

Answer the question using ONLY the retrieved customer notes.
Do not use outside knowledge or make assumptions.

If the notes do not contain enough information, respond exactly:

"I don't have enough information from the retrieved customer notes."

Retrieved Customer Notes:
{context}

Question:
{question}

Answer:
"""

    answer = generate_answer(prompt)

    return {
        "selected_tool": tool_result["tool"],
        "tool_result": tool_result,
        "answer": answer
    }

###### Section 10: End-to-End Tests

In [0]:
test_questions = [
    "How many customer notes are there?",
    "Why are customers cancelling service?"
]

for question in test_questions:
    result = answer_with_tools(question)

    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\nSELECTED TOOL:")
    print(result["selected_tool"])

    print("\nFINAL ANSWER:")
    print(result["answer"])


###### Notebook Summary

-  Loaded shared endpoint, index, embedding-model, LLM, and table configuration.

-  Connected to the existing Vector Search index.

-  Created reusable SQL and Vector Search tools with structured outputs.

-  Implemented a rule-based router for tool selection.

-  Answered count questions directly from Delta Lake.

-  Generated grounded LLM responses for semantic questions using retrieved customer-note context.

-  Unit-tested the individual tools, router, and end-to-end workflow.

###### Key Learnings

- Used Databricks Vector Search to retrieve semantically similar historical customer notes info.

- Built a grounded Retrieval-Augmented Generation (RAG) pipeline using Vector Search and an LLM.

- Developed a rule-based router that selects either the SQL count tool or the Vector Search tool based on the question pattern.

- Structured tool outputs make agent orchestration, debugging, and evaluation easier.

- Direct structured-data questions do not always require an embedding model, Vector Search, or LLM.

###### Notebook Conclusion

- In this notebook, we created reusable SQL and Vector Search tools and implemented a rule-based router that selects an appropriate tool based on the user’s question. Count questions are answered efficiently from the Delta table, while semantic questions retrieve relevant customer-note context and use an LLM to generate a grounded response.

- In the next notebook, the fixed routing logic will be replaced with LLM-based tool selection to demonstrate Agentic RAG.

###### Next Notebook

14_Agentic_RAG

The purpose of this notebook is to replace rule-based routing with LLM-based tool selection and compare fixed RAG, deterministic routing, and Agentic RAG behavior.